# 13 评分漂移检测与校准 (score_drift + calibration)

覆盖 ScoreDriftDetector / ModelCalibrator / OverduePredictor。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from hscredit import (germancredit, init_setting, WOEEncoder, LogisticRegression,
    OptimalBinning, overdue_prediction_report)
from hscredit.core.models.score_drift import ScoreDriftDetector
from hscredit.core.models.calibration import ModelCalibrator
from hscredit.core.metrics import ks, auc, psi
init_setting()
np.random.seed(42)
df = germancredit()
target = 'creditability'
num_feats = df.select_dtypes('number').columns.drop(target).tolist()
X = df[num_feats]; y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
woe = WOEEncoder(max_n_bins=5)
X_woe_tr = woe.fit_transform(X_tr, y_tr)
X_woe_te = woe.transform(X_te)
lr = LogisticRegression(max_iter=1000); lr.fit(X_woe_tr, y_tr)
y_prob_tr = lr.predict_proba(X_woe_tr)[:,1]
y_prob_te = lr.predict_proba(X_woe_te)[:,1]
print('准备完成, KS:', round(ks(y_te,y_prob_te),4))

## 1. ScoreDriftDetector — 评分漂移检测

In [ ]:
detector = ScoreDriftDetector(n_bins=10, psi_threshold=0.1)
detector.fit(y_prob_tr)  # 以训练集概率为基准

In [ ]:
# 检测测试集漂移
result = detector.detect(y_prob_te)
print(f'PSI: {result["psi"]:.4f}')
print(f'漂移状态: {result["drift_status"]}')
result.get('detail', pd.DataFrame())

In [ ]:
# 模拟更强漂移
y_prob_drifted = np.clip(y_prob_te + np.random.normal(0.15, 0.05, len(y_prob_te)), 0, 1)
result2 = detector.detect(y_prob_drifted)
print(f'强漂移 PSI: {result2["psi"]:.4f}, 状态: {result2["drift_status"]}')

## 2. ModelCalibrator — 模型概率校准

In [ ]:
# 等频分箱校准
calibrator = ModelCalibrator(method='isotonic', n_bins=10)
calibrator.fit(y_prob_tr, y_tr)
y_prob_cal = calibrator.transform(y_prob_te)
print(f'校准前 - KS: {ks(y_te,y_prob_te):.4f}, AUC: {auc(y_te,y_prob_te):.4f}')
print(f'校准后 - KS: {ks(y_te,y_prob_cal):.4f}, AUC: {auc(y_te,y_prob_cal):.4f}')

# 查看校准效果
print('\n校准前后概率对比:')
pd.DataFrame({'原始概率': y_prob_te[:8].round(4), '校准概率': y_prob_cal[:8].round(4)})

In [ ]:
# Platt Scaling 校准
cal_platt = ModelCalibrator(method='sigmoid')
cal_platt.fit(y_prob_tr, y_tr)
y_prob_platt = cal_platt.transform(y_prob_te)
print(f'Platt校准 KS: {ks(y_te,y_prob_platt):.4f}')

## 3. OverduePredictor — 逾期数据预测

In [ ]:
# 构造 mob 数据
np.random.seed(42)
n_users = 300
df_mob = pd.DataFrame({
    'user_id': range(n_users),
    'apply_month': np.random.choice(['2023-01','2023-02','2023-03'], n_users),
    'mob': np.random.randint(1, 12, n_users),
    'fpd15': np.random.binomial(1, 0.08, n_users),
    'fpd30': np.random.binomial(1, 0.12, n_users),
    'fpd60': np.random.binomial(1, 0.05, n_users),
    'amount': np.random.uniform(5000, 100000, n_users),
})
from hscredit import OverduePredictor
pred = OverduePredictor(target_cols=['fpd15','fpd30'], mob_col='mob', max_mob=12)
pred.fit(df_mob)
print('预测补全中...')
df_pred = pred.predict(df_mob)
print(df_pred[['user_id','mob','fpd15','fpd30']].head(8))

In [ ]:
# 逾期预测报告
rep = overdue_prediction_report(df_mob, target_cols=['fpd15','fpd30'],
    mob_col='mob', cohort_col='apply_month')
print(rep.head())